Notebook describes how to extract all required data from the BACI original files and concatenate evrything in a single SQL database containing all relevant information. Note that generating this database requires minimum 12GB of RAM. Under that threshold you will most likely hit an unavoidable MemoryError. Even at 12GB of RAM, there are a lot of data to handle so your laptop will be slowed down.

In [1]:
import pandas as pd
import sqlite3
import sys
sys.path.append('C://Users/11max/PycharmProjects/Regioinvent/src/')
import regioinvent
import re

Start by reading all the BACI files to include in the SQL database. Here we selected the v202601 version of BACI for the 5 latest available years

In [2]:
# baci2018 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS17/BACI_HS17_Y2018_V202601.csv', low_memory=False)
# baci2019 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS17/BACI_HS17_Y2019_V202601.csv', low_memory=False)
baci2020 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS17/BACI_HS17_Y2020_V202601.csv', low_memory=False)
baci2021 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS17/BACI_HS17_Y2021_V202601.csv', low_memory=False)
baci2022 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS17/BACI_HS17_Y2022_V202601.csv', low_memory=False)
baci2023 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS17/BACI_HS17_Y2023_V202601.csv', low_memory=False)
baci2024 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS17/BACI_HS17_Y2024_V202601.csv', low_memory=False)

Concatenate everything in a single dataframe

In [3]:
baci = pd.concat([baci2020, baci2021, baci2022, baci2023, baci2024])
baci = baci.reset_index().drop('index', axis=1)
baci.columns = ['refYear', 'exporter', 'importer', 'cmdCode', 'value (1,000 USD)', 'quantity (t)']
# remove NaN values for quantity
baci = baci[~baci.loc[:, 'quantity (t)'].isna()]

Do some formatting

In [4]:
baci.loc[:, 'quantity'] = baci.loc[:, 'quantity (t)'].astype(float)
baci.cmdCode = baci.cmdCode.astype(str)
baci.loc[:, 'cmdCode'] = ['0'+i if len(i) == 5 else i for i in baci.cmdCode]
baci = baci.drop('quantity (t)', axis=1)
baci = baci.rename(columns={'quantity': 'quantity (t)'})

Here we translate the country codes of BACI (by default these are numbers) to the more conventional ISO 3-letter country codes

In [5]:
country_codes = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS17/country_codes_V202601.csv', low_memory=False).set_index('country_code')
country_codes = country_codes.loc[:, 'country_iso3'].to_dict()
# restore ISO3 codes instead of numbers
baci.importer = [country_codes[i] for i in baci.importer] 
baci.exporter = [country_codes[i] for i in baci.exporter]

Then we ensure that HS codes are all untouched (there might be shenanigans happening while passing hrough the csv format)

In [ ]:
# add zeros in front of the agricutural HS commodity codes
baci.cmdCode = baci.cmdCode.astype(str)
# only HS6 codes so no need to add zeros in front of HS4 codes, because they are created later
baci.loc[:, 'cmdCode'] = ['0'+i if len(i) == 5 else i for i in baci.cmdCode]

For a few commodities, the HS17 classification is limited and we rely instead of the the HS22 data, even though it is limited to 2022 only

In [5]:
# the commodities for which we rely on HS22
with_baci_22 = ['290341', '290343', '290349', '854142', '854143', '854159', '441881', '441882', '441883']

Load the relevant HS22 BACI files and format the data

In [6]:
bacihs222022 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS22/BACI_HS22_Y2022_V202601.csv', low_memory=False)
bacihs222023 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS22/BACI_HS22_Y2023_V202601.csv', low_memory=False)
bacihs222024 = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS22/BACI_HS22_Y2024_V202601.csv', low_memory=False)
bacihs22 = pd.concat([bacihs222022, bacihs222023, bacihs222024])
bacihs22 = bacihs22.reset_index().drop('index', axis=1)

# rename columns
bacihs22.columns = ['refYear', 'exporter', 'importer', 'cmdCode', 'value (1,000 USD)', 'quantity (t)']
# remove NaN values for quantity
bacihs22 = bacihs22[~bacihs22.loc[:, 'quantity (t)'].isna()]
# change data formats
bacihs22.loc[:, 'quantity'] = bacihs22.loc[:, 'quantity (t)'].astype(float)
bacihs22.cmdCode = bacihs22.cmdCode.astype(str)
bacihs22 = bacihs22.drop('quantity (t)', axis=1)
bacihs22 = bacihs22.rename(columns={'quantity': 'quantity (t)'})
country_codes = pd.read_csv('C://Users/11max/Desktop/Work/Databases/BACI/v202601/HS22/country_codes_V202601.csv', low_memory=False).set_index('country_code')
country_codes = country_codes.loc[:, 'country_iso3'].to_dict()
# restore ISO3 codes instead of numbers
bacihs22.importer = [country_codes[i] for i in bacihs22.importer] 
bacihs22.exporter = [country_codes[i] for i in bacihs22.exporter]

Add this HS22 data, only for the relevant commodities, to the previously extracted HS17 data

In [9]:
trade_data = pd.concat([baci, bacihs22.loc[bacihs22.cmdCode.isin(with_baci_22)]])
trade_data = trade_data.reset_index().drop('index', axis=1)

Then we simply export the whole databas to an SQL database.

In [12]:
conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/baci_trade_data_full_version.db')
trade_data.set_index('cmdCode').to_sql('International trade data', conn)

55564816

This extracted every trade flows of HS17 from the selected files, and the few extra commodities from HS22. The resulting databases is pretty big. For final use within regioinvent, we do not need all these trade flows. We only need to trade flows which are connected for ecoinvent processes. However, the full BACI version will come in handy later when treated domestic production estimates.

### Filter out commodities we don't want in the trade database

In [15]:
# load the codes of commodities in ecoinvent
# regio391 = regioinvent.Regioinvent(bw_project_name='ecoinvent3.9.1', ecoinvent_database_name='ecoinvent-3.9.1-cutoff', ecoinvent_version='3.9.1')
regio310 = regioinvent.Regioinvent(bw_project_name='ecoinvent3.10', ecoinvent_database_name='ecoinvent-3.10.1-cutoff', ecoinvent_version='3.10')
regio311 = regioinvent.Regioinvent(bw_project_name='ecoinvent3.11', ecoinvent_database_name='ecoinvent-3.11-cutoff', ecoinvent_version='3.11')
regio312 = regioinvent.Regioinvent(bw_project_name='ecoinvent3.12', ecoinvent_database_name='ecoinvent-3.12-cutoff', ecoinvent_version='3.12')
# regroup all required HS codes in a single set
all_needed_cmdCodes = set(list(regio310.eco_to_hs_class.values()) + list(regio311.eco_to_hs_class.values()) + list(regio312.eco_to_hs_class.values()))

Some of the mappings with ecoinvent are not with HS6 codes, but rather with HS4 codes (more aggregated codes). We need to recreate the data for these codes, by simply aggregating the HS6 codes velonging to the HS4 codes.

In [18]:
hs6codes = list(set([i for i in all_needed_cmdCodes if len(i) == 6]))
hs4codes = list(set([i for i in all_needed_cmdCodes if len(i) == 4]))
all_cmdcodes = set(trade_data.cmdCode)

for hs4code in hs4codes:
    codes_to_add = [i for i in all_cmdcodes if re.findall(rf'^{hs4code}', i)]
    for code in codes_to_add:
        hs6codes.append(code)
# filtering will speed up next step
trade_data = trade_data.loc[trade_data.cmdCode.isin(hs6codes)]

hs4codes = set([i for i in all_needed_cmdCodes if len(i) == 4])

all_cmdcodes = set(trade_data.cmdCode)

for hs4code in hs4codes:
    # identify hs6codes being part of the selected hs4code
    hs6codes_to_aggregate = [i for i in all_cmdcodes if re.findall(rf'^{hs4code}', i)]
    # filter the dataframe
    df = trade_data.loc[trade_data.cmdCode.isin(hs6codes_to_aggregate)].copy()
    # groupby values for selected hs4code
    df = df.groupby(by=['refYear', 'exporter', 'importer', 'value (1,000 USD)', 'quantity (t)']).sum()
    # change the cmdCode
    df.loc[:, 'cmdCode'] = hs4code
    # concat with trade_data
    trade_data = pd.concat([trade_data, df.reset_index()])

Finally, there is a clear mistake in BACI data for the export of natural gas (271121) from Mozambia (MOZ) to South Africa (ZAF) in 2021. We correct it here by taking the value from year 2020. The dollar values are similar so it's not a bad approximation. Mistake also spreads to commodity overall petroleum gases (2711) so we correct it as well.

In [4]:
trade_data.loc[(trade_data.cmdCode == '271121') & 
(trade_data.exporter == 'MOZ') & 
(trade_data.importer == 'ZAF') &
(trade_data.refYear == 2021), 'quantity (t)'] = trade_data.loc[(trade_data.cmdCode == '271121') & 
(trade_data.exporter == 'MOZ') & 
(trade_data.importer == 'ZAF') &
(trade_data.refYear == 2020), 'quantity (t)'].iloc[0]

trade_data.loc[(trade_data.cmdCode == '2711') & 
(trade_data.exporter == 'MOZ') & 
(trade_data.importer == 'ZAF') &
(trade_data.refYear == 2021), 'quantity (t)'] = trade_data.loc[(trade_data.cmdCode.isin(['271111','271121'])) & 
(trade_data.exporter == 'MOZ') & 
(trade_data.importer == 'ZAF') &
(trade_data.refYear == 2021), 'quantity (t)'].sum()

trade_data = trade_data.groupby(['cmdCode','refYear','exporter','importer']).sum().reset_index()

We write this filtered version of the BACI trade database (tailored towards connection wtih ecoinvent) in another SQL database

In [20]:
new_data = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/trade_data.db')
# cursor = new_data.cursor()
# cursor.execute('DROP TABLE [International trade data]')
trade_data.sort_values(by=['cmdCode', 'quantity (t)'], ascending=[True, False]).set_index('cmdCode').to_sql('Original trade data', new_data)

11952153